In [ ]:
import numpy as np
import torch
import torch.optim as optim
import logging
import matplotlib.pyplot as plt
from argparse import ArgumentParser
from torch.autograd import Variable
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.distributions import Normal, OneHotCategorical
import torch.nn.utils as nn_utils
from sklearn.metrics import r2_score
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(123456)
np.random.seed(123456)


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)        
class MDN(nn.Module):
    def __init__(self, n_hidden, n_gaussians,n_hidden_layers,bn, dout, dout_value):
        super(MDN, self).__init__()                
        layers = [nn.Linear(41, n_hidden), nn.Tanh()] # 41 = Input features
        if bn == 1:
            layers.append(nn.BatchNorm1d(n_hidden))    
        for _ in range(n_hidden_layers - 2):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())
        if dout==1:
            layers.append(nn.Dropout(dout_value))
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())   
        else:
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())       
        self.z_h = nn.Sequential(*layers)                        
        self.z_pi = nn.Linear(n_hidden, n_gaussians)
        self.z_sigma = nn.Linear(n_hidden, n_gaussians)
        self.z_mu = nn.Linear(n_hidden, n_gaussians)  
    def forward(self, x):
        z_h = self.z_h(x)
        pi = nn.functional.softmax(self.z_pi(z_h), -1)
        sigma = torch.exp(self.z_sigma(z_h))+ 1e-8
        sigma = torch.clamp(sigma, min=1e-4)
        mu = 0 + (1 - 0) * (torch.tanh(self.z_mu(z_h)) + 1) / 2
        return pi, sigma, mu    
oneDivSqrtTwoPI = 1.0 / np.sqrt(2.0*np.pi) 
def gaussian_distribution(y, mu, sigma):
    result = (y.expand_as(mu) - mu) * torch.reciprocal(sigma)
    result = -0.5 * (result * result)
    return (torch.exp(result) * torch.reciprocal(sigma)) * oneDivSqrtTwoPI
def mdn_loss_fn(pi, sigma, mu, y):
    result = gaussian_distribution(y, mu, sigma) * pi
    result1 = torch.sum(result, dim=1)
    result2 = -torch.log(result1+1e-12)
    return torch.mean(result2)


model = MDN(n_hidden=70, n_gaussians=3,n_hidden_layers=8,bn=0, dout=0, dout_value=0.24734) 
model.eval()
optimizer = optim.Adam(model.parameters(), lr=0.00007)
PATH = "checkpoint/model-319.pt" 
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])


data_x = np.load('xlo_test.npy')[:,:]
data_y = np.load('ylo_test.npy')
data_y[:,1]=data_y[:,1]+data_y[:,3]  
data_x_np = torch.tensor(data_x, dtype=torch.float)
data_y_np = torch.tensor(data_y, dtype=torch.float)[:,4:5]  
pi_variable, sigma_variable, mu_variable = model(data_x_np)
loss = mdn_loss_fn(pi_variable, sigma_variable, mu_variable, data_y_np)
weighted_average_mu = torch.sum(pi_variable * mu_variable, dim=-1)
weighted_sigma = torch.sqrt(torch.sum(pi_variable * (sigma_variable ** 2 + (mu_variable - weighted_average_mu.unsqueeze(-1)) ** 2), dim=-1))
weighted_average_mu_np = weighted_average_mu.detach().numpy()
weighted_sigma_np = weighted_sigma.detach().numpy()
data_y_np = data_y_np.detach().numpy()

np.random.seed(42)
true = data_y_np[:, 0]
pred = weighted_average_mu_np[:]  
metric = (true - pred)  
x_bins = np.linspace(0, 1, 50)  
y_bins = np.linspace(-1, 1, 50)
H, x_edges, y_edges = np.histogram2d(true, metric, bins=[x_bins, y_bins])
plt.figure(figsize=(8, 6))
plt.pcolormesh(
    x_edges, 
    y_edges, 
    H.T, 
    cmap="viridis", 
    edgecolors="white",  
    linewidth=0.5,
    vmin=0, 
    vmax=5
)
cbar=plt.colorbar(label="Density")
cbar.ax.tick_params(labelsize=20) 
cbar.set_label("Density", fontsize=20)
plt.xlabel("True Value",fontsize=20, fontname='Times New Roman')
plt.ylabel("True - Predicted Weighted Mean Value",fontsize=20, fontname='Times New Roman')
plt.xticks(fontsize=20, fontname='Times New Roman')
plt.yticks(fontsize=20, fontname='Times New Roman') 
plt.tight_layout()
plt.savefig('5f.png', dpi=600)